In [2]:
using Symbolics, OffsetArrays
using SymbolicUtils
using LinearAlgebra


In [3]:
@variables A[1:1,0:2] B[1:1,0:2] C[1:1,0:2] D[1:1,0:2] a[1:1] c[1:1] w[0:2] u[0:2]

8-element Vector{Symbolics.Arr{Num}}:
 A[1:1,0:2]
 B[1:1,0:2]
 C[1:1,0:2]
 D[1:1,0:2]
 a[1:1]
 c[1:1]
 w[0:2]
 u[0:2]

In [4]:
eq1 = dot(A,u) + dot(B,w) ~ a[1]
eq2 = dot(C,u) + dot(D,w) ~ c[1]

C[1, 0]*u[0] + C[1, 1]*u[1] + C[1, 2]*u[2] + D[1, 0]*w[0] + D[1, 1]*w[1] + D[1, 2]*w[2] ~ c[1]

In [5]:
eqs = [eq1, eq2]
sol = solve_for(eqs,[u[0], w[0]])

2-element Vector{SymbolicUtils.BasicSymbolicImpl.var"typeof(BasicSymbolicImpl)"{SymReal}}:
 (-a[1] + (B[1, 0]*(-c[1] + (-(-a[1] + A[1, 1]*u[1] + A[1, 2]*u[2] + B[1, 1]*w[1] + B[1, 2]*w[2])*C[1, 0]) / A[1, 0] + C[1, 1]*u[1] + C[1, 2]*u[2] + D[1, 1]*w[1] + D[1, 2]*w[2])) / (-D[1, 0] + (B[1, 0]*C[1, 0]) / A[1, 0]) + A[1, 1]*u[1] + A[1, 2]*u[2] + B[1, 1]*w[1] + B[1, 2]*w[2]) / (-A[1, 0])
 (-c[1] + (-(-a[1] + A[1, 1]*u[1] + A[1, 2]*u[2] + B[1, 1]*w[1] + B[1, 2]*w[2])*C[1, 0]) / A[1, 0] + C[1, 1]*u[1] + C[1, 2]*u[2] + D[1, 1]*w[1] + D[1, 2]*w[2]) / (-D[1, 0] + (B[1, 0]*C[1, 0]) / A[1, 0])

In [6]:
@variables dx Kbar[0:1] Fmid[0:1] Gmid[0:1] dt F[1:1] G[1:1] Fx[1:1] Gx[1:1] w_old[0:2]

# we take the discretized eqs for the internal points and copy last known point for 
# Kbar, F, G to the midpoint 1/2 (for i=0), copied that from 3/2 (for i=1, that is the 1st point)
# and the same for w_old[0] from w_old[1]

subs_i1_step1 = Dict(
    # 1st eq
    A[1,0] => Kbar[0]/dx^2,
    A[1,1] => -(Kbar[1]+Kbar[0])/dx^2,
    A[1,2] => Kbar[1]/dx^2,
    B[1,0] => -Kbar[0]/dx^2,
    B[1,1] => (Kbar[1]+Kbar[0])/dx^2,
    B[1,2] => -Kbar[1]/dx^2,
    a[1]=>0,
    # 2nd eq
    C[1,0] => dt*Fmid[0]*Gmid[0]/2,
    C[1,1] => -dt*(Fmid[1]*Gmid[1]/2-Fmid[0]*Gmid[0]/2-dx*(Fx[1]*G[1]+F[1]*Gx[1])),
    C[1,2] => -dt*Fmid[1]*Gmid[1]/2,
    D[1,0] => -(1+dt*Fmid[0]*Gmid[0])/2,
    D[1,1] => dt*(Fmid[1]*Gmid[1]/2-Fmid[0]*Gmid[0]/2-dx*(Fx[1]*G[1]+F[1]*Gx[1])),
    D[1,2] => (1+dt*Fmid[1]*Gmid[1])/2,
    c[1] => (w_old[2] - w_old[0])/2  
)

Dict{Num, Real} with 14 entries:
  a[1]    => 0
  D[1, 2] => (1 + Fmid[1]*Gmid[1]*dt) / 2
  A[1, 1] => (-Kbar[0] - Kbar[1]) / (dx^2)
  C[1, 1] => -(-(1//2)*Fmid[0]*Gmid[0] + (1//2)*Fmid[1]*Gmid[1] - (F[1]*Gx[1] +…
  c[1]    => (-w_old[0] + w_old[2]) / 2
  B[1, 1] => (Kbar[0] + Kbar[1]) / (dx^2)
  B[1, 2] => (-Kbar[1]) / (dx^2)
  D[1, 1] => (-(1//2)*Fmid[0]*Gmid[0] + (1//2)*Fmid[1]*Gmid[1] - (F[1]*Gx[1] + …
  C[1, 0] => (1//2)*Fmid[0]*Gmid[0]*dt
  A[1, 0] => Kbar[0] / (dx^2)
  D[1, 0] => (-1 - Fmid[0]*Gmid[0]*dt) / 2
  A[1, 2] => Kbar[1] / (dx^2)
  C[1, 2] => (-1//2)*Fmid[1]*Gmid[1]*dt
  B[1, 0] => (-Kbar[0]) / (dx^2)

In [21]:
subs_i1_step2 = Dict(
    Fmid[0] => Fmid[1],
    Gmid[0] => Gmid[1],
    Kbar[0] => Kbar[1],
    w_old[0] => w_old[1]
)

Dict{Num, Num} with 4 entries:
  Gmid[0]  => Gmid[1]
  w_old[0] => w_old[1]
  Kbar[0]  => Kbar[1]
  Fmid[0]  => Fmid[1]

In [22]:
sol1 = substitute(sol, subs_i1_step1)
sol2 = substitute(sol1, subs_i1_step2)

2-element Vector{SymbolicUtils.BasicSymbolicImpl.var"typeof(BasicSymbolicImpl)"{SymReal}}:
 (-(dx^2)*((-2Kbar[1]*u[1]) / (dx^2) + (-Kbar[1]*w[2]) / (dx^2) + (2Kbar[1]*w[1]) / (dx^2) + (-((w_old[1] - w_old[2]) / 2 + ((-1//2)*Fmid[1]*Gmid[1]*dt*(dx^2)*((-2Kbar[1]*u[1]) / (dx^2) + (-Kbar[1]*w[2]) / (dx^2) + (2Kbar[1]*w[1]) / (dx^2) + (Kbar[1]*u[2]) / (dx^2))) / Kbar[1] - (1//2)*Fmid[1]*Gmid[1]*dt*u[2] + (1//2)*(1 + Fmid[1]*Gmid[1]*dt)*w[2] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1])*Kbar[1]) / ((-(1//2)*Fmid[1]*Gmid[1]*dt + (1 + Fmid[1]*Gmid[1]*dt) / 2)*(dx^2)) + (Kbar[1]*u[2]) / (dx^2))) / Kbar[1]
 ((w_old[1] - w_old[2]) / 2 + ((-1//2)*Fmid[1]*Gmid[1]*dt*(dx^2)*((-2Kbar[1]*u[1]) / (dx^2) + (-Kbar[1]*w[2]) / (dx^2) + (2Kbar[1]*w[1]) / (dx^2) + (Kbar[1]*u[2]) / (dx^2))) / Kbar[1] - (1//2)*Fmid[1]*Gmid[1]*dt*u[2] + (1//2)*(1 + Fmid[1]*Gmid[1]*dt)*w[2] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1]) / (-(1//2)*Fmid[1]*Gmid[1]*

In [8]:
# copied from https://discourse.julialang.org/t/help-with-symbolics-how-to-get-coefficients-and-simple-roots/68180/3
function mycoeffs(f,var,order)
    if order == 0
        return substitute(f,Dict(var=>0))
    end
    D=Differential(var^order)
    newf=expand_derivatives(D(f))
    return substitute(newf,Dict(var=>0))
end

mycoeffs (generic function with 1 method)

In [9]:
wx_1 = (w[2]-w[0])/(2*dx)

(-w[0] + w[2]) / (2dx)

In [23]:
wx_1_b = substitute(wx_1, w[0]=>sol2[2])

(w[2] + (-((w_old[1] - w_old[2]) / 2) - (((-1//2)*Fmid[1]*Gmid[1]*dt*(dx^2)*((-2Kbar[1]*u[1]) / (dx^2) + (-Kbar[1]*w[2]) / (dx^2) + (2Kbar[1]*w[1]) / (dx^2) + (Kbar[1]*u[2]) / (dx^2))) / Kbar[1]) + (1//2)*Fmid[1]*Gmid[1]*dt*u[2] - (1//2)*(1 + Fmid[1]*Gmid[1]*dt)*w[2] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1]) / (-(1//2)*Fmid[1]*Gmid[1]*dt + (1 + Fmid[1]*Gmid[1]*dt) / 2)) / (2dx)

In [11]:
#wx_1_c = substitute(wx_1_b, subs_i1_step1)

In [24]:
#wx_1_b = expand(substitute(wx_1_b, subs_i1_step1))
#wx_1_c = substitute(wx_1_c, u[1]=>ubarL)

In [25]:
wx_1_b_w1_coeff = simplify(mycoeffs(wx_1_b, w[1],1))

(Fmid[1]*Gmid[1]*dt + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx) / dx

In [26]:
wx_1_b_w2_coeff = simplify(mycoeffs(wx_1_b, w[2],1))

((1//2) - (1//2)*Fmid[1]*Gmid[1]*dt - (1//2)*(1 + Fmid[1]*Gmid[1]*dt)) / dx

In [27]:
wx_1_b_u1_coeff = simplify(mycoeffs(wx_1_b, u[1],1))

(-Fmid[1]*Gmid[1]*dt - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx) / dx

In [28]:
wx_1_b_u2_coeff = simplify(mycoeffs(wx_1_b, u[2],1))

(Fmid[1]*Gmid[1]*dt) / dx

In [29]:
the_rest = simplify(substitute(wx_1_b, Dict(w[1]=>0, w[2]=>0, u[1]=>0, u[2]=>0)))
print(the_rest)

(-w_old[1] + w_old[2]) / ((2//1)*dx)

In [30]:
"""
    for i=N we cannot use N explicitly as an index, so we use the expressions from before
    and assume that N=2 and solve for that
"""

"    for i=N we cannot use N explicitly as an index, so we use the expressions from before\n    and assume that N=2 and solve for that\n"

In [32]:
# the map is 0,1,2 -> N-1, N, N+1, with N+1 outside on the right (ghost)
# for the midpoints for i=1 we had e.g. Fmid[0] on the left of i=1
# here we have Fmid[1] on the right of i=N, so we copy Fmid[1] from Fmid[0]
# for w_old[2] is copied from w_old[1]
subs_iN_step2 = Dict(
    Fmid[1] => Fmid[0],
    Gmid[1] => Gmid[0],
    Kbar[1] => Kbar[0],
    w_old[2] => w_old[1]
)

Dict{Num, Num} with 4 entries:
  Gmid[1]  => Gmid[0]
  Kbar[1]  => Kbar[0]
  Fmid[1]  => Fmid[0]
  w_old[2] => w_old[1]

In [33]:
solN = solve_for(eqs,[u[2], w[2]])
solN = substitute(solN, subs_i1_step1)
solN = substitute(solN, subs_iN_step2)

2-element Vector{SymbolicUtils.BasicSymbolicImpl.var"typeof(BasicSymbolicImpl)"{SymReal}}:
 (-(dx^2)*((-((w_old[0] - w_old[1]) / 2 + ((1//2)*Fmid[0]*Gmid[0]*dt*(dx^2)*((2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0] + (1//2)*Fmid[0]*Gmid[0]*dt*u[0] + (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)*w[0] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1])*Kbar[0]) / (((1//2)*Fmid[0]*Gmid[0]*dt + (-1 - Fmid[0]*Gmid[0]*dt) / 2)*(dx^2)) + (2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0]
 ((w_old[0] - w_old[1]) / 2 + ((1//2)*Fmid[0]*Gmid[0]*dt*(dx^2)*((2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0] + (1//2)*Fmid[0]*Gmid[0]*dt*u[0] + (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)*w[0] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1]) / ((1//2)*Fmid[0]*Gmid[0]*d

In [35]:
wx_N = (w[2]-w[0])/(2*dx)
wx_N_b = substitute(wx_N, w[2]=>solN[2])
#wx_N_c = substitute(wx_N_b, subs_i1_step1)
#wx_N_c = expand(substitute(wx_N_c, subs_iN_step2))

(-w[0] + ((w_old[0] - w_old[1]) / 2 + ((1//2)*Fmid[0]*Gmid[0]*dt*(dx^2)*((2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0] + (1//2)*Fmid[0]*Gmid[0]*dt*u[0] + (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)*w[0] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1]) / ((1//2)*Fmid[0]*Gmid[0]*dt + (-1 - Fmid[0]*Gmid[0]*dt) / 2)) / (2dx)

In [36]:
wx_N_b_u0_coeff = simplify(mycoeffs(wx_N_b, u[0],1))
print(wx_N_b_u0_coeff)

(-Fmid[0]*Gmid[0]*dt) / dx

In [37]:
wx_N_b_u1_coeff = simplify(mycoeffs(wx_N_b, u[1],1))
print(wx_N_b_u1_coeff)

(-Fmid[0]*Gmid[0]*dt + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx) / (-dx)

In [38]:
wx_N_b_w0_coeff = simplify(mycoeffs(wx_N_b, w[0],1))
print(wx_N_b_w0_coeff)

((1//2) - (1//2)*Fmid[0]*Gmid[0]*dt + (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)) / (-dx)

In [39]:
wx_N_b_w1_coeff = simplify(mycoeffs(wx_N_b, w[1],1))
print(wx_N_b_w1_coeff)

(Fmid[0]*Gmid[0]*dt - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx) / (-dx)

In [40]:
the_rest_iN = simplify(substitute(wx_N_b, Dict(w[1]=>0, w[0]=>0, u[1]=>0, u[0]=>0)))
print(the_rest_iN)

(w_old[0] - w_old[1]) / ((-2//1)*dx)

In [42]:
"""
for [u_x-w_x]_{i=N}
"""

ux_minus_wx_N = (u[2]-u[0])/(2*dx) - (w[2]-w[0])/(2*dx)
ux_minus_wx_N_b = substitute(ux_minus_wx_N, Dict(u[2]=>solN[1], w[2]=>solN[2]))

(w[0] + (-((w_old[0] - w_old[1]) / 2) - (((1//2)*Fmid[0]*Gmid[0]*dt*(dx^2)*((2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0]) - (1//2)*Fmid[0]*Gmid[0]*dt*u[0] - (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)*w[0] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1]) / ((1//2)*Fmid[0]*Gmid[0]*dt + (-1 - Fmid[0]*Gmid[0]*dt) / 2)) / (2dx) + (-u[0] + (-(dx^2)*((-((w_old[0] - w_old[1]) / 2 + ((1//2)*Fmid[0]*Gmid[0]*dt*(dx^2)*((2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0] + (1//2)*Fmid[0]*Gmid[0]*dt*u[0] + (1//2)*(-1 - Fmid[0]*Gmid[0]*dt)*w[0] + (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*u[1] - (F[1]*Gx[1] + Fx[1]*G[1])*dt*dx*w[1])*Kbar[0]) / (((1//2)*Fmid[0]*Gmid[0]*dt + (-1 - Fmid[0]*Gmid[0]*dt) / 2)*(dx^2)) + (2Kbar[0]*w[1]) / (dx^2) + (Kbar[0]*u[0]) / (dx^2) + (-2Kbar[0]*u[1]) / (dx^2) + (-Kbar[0]*w[0]) / (dx^2))) / Kbar[0]) / (2dx)

In [43]:
ux_minus_wx_N_b_u0_coeff = simplify(mycoeffs(ux_minus_wx_N_b, u[0],1))
print(ux_minus_wx_N_b_u0_coeff)

-1 / dx

In [44]:
ux_minus_wx_N_b_u1_coeff = simplify(mycoeffs(ux_minus_wx_N_b, u[1],1))
print(ux_minus_wx_N_b_u1_coeff)

1 / dx

In [45]:
ux_minus_wx_N_b_w0_coeff = simplify(mycoeffs(ux_minus_wx_N_b, w[0],1))
print(ux_minus_wx_N_b_w0_coeff)

1 / dx

In [46]:
ux_minus_wx_N_b_w1_coeff = simplify(mycoeffs(ux_minus_wx_N_b, w[1],1))
print(ux_minus_wx_N_b_w1_coeff)

-1 / dx

In [47]:
ux_minus_wx_N_B_the_rest = simplify(substitute(ux_minus_wx_N_b, Dict(w[1]=>0, w[0]=>0, u[1]=>0, u[0]=>0)))
print(ux_minus_wx_N_B_the_rest)

0